# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates loading and exploring a FAIR^2 dataset using the `mlcroissant` library, referencing all schema elements by their `@id`.

### Dataset Source
The dataset is defined via a Croissant schema at the following URL (FAIR^2 schema):

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. The metadata describes the dataset context, record sets, fields, and data files.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant Dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset, metadata, and examine basic information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")


## 2. Data Overview

Review available record sets, fields, and their IDs.

Below, we enumerate all record sets and display their basic details referencing each by its `@id`.

In [ ]:
# List all record sets with their @id and names
record_sets = list(dataset.record_sets)

print(f"Found {len(record_sets)} record set(s):\n")
for i, rs in enumerate(record_sets):
    print(f"{i+1}. @id: {rs.id}")
    print(f"   name: {getattr(rs, 'name', None)}")
    print(f"   description: {getattr(rs, 'description', None)}\n")
    
    # List all fields within this record set (by @id)
    print("   Fields:")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"      - @id: {field.id} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")
    else:
        print("      [No fields listed]")
    print()

## 3. Data Extraction

Load data from each record set into a Pandas DataFrame. All access uses `@id` for record sets and fields.

**Note:** Since the number and identity of record sets may vary, we collect all and create a DataFrame for each, keyed by their `@id`. Each field within a record set is referenced by its `@id`.

In [ ]:
# Collect record set @ids
record_set_ids = [rs.id for rs in record_sets]
print("Loaded record set @ids:", record_set_ids)

# Load data for each record set, using the @id as key
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    # Preview columns
    print(f"\nFirst few columns for record set @id {record_set_id}:")
    print(df.columns.tolist())
    print(df.head(2).to_string(index=False))

## 4. Exploratory Data Analysis (EDA)

Apply typical data exploration steps: filtering numeric fields, normalizing, grouping, and handling missing data.

The following block demonstrates analysis for the **first record set** (edit as desired to target another by its `@id`).

**All fields must be referenced by their `@id`.**

In [ ]:
# Pick the first available record set
if len(record_set_ids) == 0:
    raise ValueError("No record sets found in this dataset.")
active_record_set_id = record_set_ids[0]
df = dataframes[active_record_set_id]

print(f"Examining DataFrame for record set @id: {active_record_set_id}")

# Display all field @ids (columns)
print("Available field @ids:")
for col in df.columns:
    print(f" - {col}")

# Try to identify a numeric (float or int) field by inspecting dtypes and field names
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_candidates:
    # If dtype not parsed, guess typical numeric field names or columns containing 'coef', 'pvalue', 'score', 'll', etc.
    numeric_candidates = [col for col in df.columns if any(substr in col.lower() for substr in ("coef", "std", "se", "value", "ll", "pval", "prob"))]

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"\nUsing numeric field @id: {numeric_field_id}")
else:
    print("No obvious numeric field identified; fill 'numeric_field_id' manually.")
    numeric_field_id = df.columns[0]  # fallback

# Set a filtering threshold (example: 0 for log-likelihood, or 0 for coefficients, etc.)
threshold = 0

filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold} (showing first 5):")
print(filtered_df.head())

# Normalize the numeric field
if not filtered_df.empty and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for {numeric_field_id} (first 5):")
    print(filtered_df[[numeric_field_id, norm_col]].head())
else:
    print("Could not normalize: Field is empty or not numeric after filtering.")

# Identify a categorical/group-by field (example: one named with 'ward', 'region', 'variable', etc.)
group_field_candidates = [col for col in df.columns if any(substr in col.lower() for substr in ("ward", "region", "cat", "group", "variable"))]
group_field_id = group_field_candidates[0] if group_field_candidates else None

if group_field_id is not None and group_field_id in filtered_df.columns:
    # Use numeric_field_id for aggregation
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No obvious group-by field found for grouping.")

## 5. Visualization

Visualize data distributions and relationships between key numeric and categorical fields using matplotlib and seaborn, referencing all column names by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the chosen numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna().astype(float), bins=20)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} in record set {active_record_set_id}")
plt.tight_layout()
plt.show()

# If a grouping field was identified, plot group means
if group_field_id is not None and group_field_id in df.columns:
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(9,4))
    sns.barplot(data=group_means, x=group_field_id, y=numeric_field_id)
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("No group-by field found for grouped visualization.")

## 6. Conclusion

We explored the structured metadata and records of this FAIR^2 dataset using the `mlcroissant` library. All references were made via `@id` for record sets and fields, ensuring reproducibility and schema traceability. The EDA and visualization steps revealed basic distributional properties of a key numeric variable and the effect of grouping by categorical fields when present.

You can extend this notebook to perform deeper statistical modeling or more advanced analysis using other fields identified above.

_Remember: Always use entity `@id`s for unambiguous referencing within the FAIR metadata ecosystem!_